# TF-FaultNet: complete three-class audio fault-diagnosis pipeline

This notebook is a Google Colab front end for the repository's canonical pipeline. It runs from environment setup through locked test evaluation without duplicating model code.

**Order:** problem/data audit -> EDA -> frozen split validation -> classical baselines -> TF-FaultNet training -> held-out evaluation -> optional ablation -> artifact summary.

**Reproducibility and leakage rules:** `data/splits.csv` is the immutable file-level 70/15/15 manifest; the test partition is not used for preprocessing, model selection, or early stopping; learned preprocessing is fit on training data only; all randomness uses seed 42. A Colab GPU runtime is strongly recommended for training.

## 0. Colab controls

Set `PROJECT_SOURCE` to a public GitHub URL, a Google Drive project folder, or leave it as `upload` to upload a ZIP containing the `Audio/` project folder. The expensive ablation stage is opt-in.

In [ ]:
PROJECT_SOURCE = "upload"  # "upload", a GitHub HTTPS URL, or a Drive folder path
PROJECT_DIR_NAME = "Audio"
MOUNT_GOOGLE_DRIVE = False
RUN_ABLATIONS = False
SEED = 42

import os
import sys
import json
import zipfile
import shutil
import subprocess
from pathlib import Path
from IPython.display import display, Markdown, Image

def run(command, cwd=None, check=True):
    """Run a project command and show its output; never hide failures."""
    print(f"$ {command}")
    result = subprocess.run(command, shell=True, cwd=cwd, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {command}")
    return result

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

WORKSPACE = Path("/content")
PROJECT_ROOT = WORKSPACE / PROJECT_DIR_NAME

if PROJECT_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise ValueError("Upload one ZIP file containing the Audio project folder.")
    with zipfile.ZipFile(zip_names[0]) as archive:
        archive.extractall(WORKSPACE)
elif PROJECT_SOURCE.startswith(("http://", "https://")):
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    run(f"git clone --depth 1 {PROJECT_SOURCE} {PROJECT_ROOT}")
elif PROJECT_SOURCE:
    PROJECT_ROOT = Path(PROJECT_SOURCE).expanduser().resolve()

if not (PROJECT_ROOT / "requirements.txt").is_file():
    raise FileNotFoundError(f"Could not find requirements.txt under {PROJECT_ROOT}")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

## 1. Install dependencies and inspect the runtime

The project pins minimum compatible versions in `requirements.txt`. Colab often already has NumPy, PyTorch, and scikit-learn; installing from the manifest makes the environment reproducible.

In [ ]:
run(f"{sys.executable} -m pip install -q -r requirements.txt", cwd=PROJECT_ROOT)
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CPU training is supported but substantially slower.")


## 2. Validate the data, taxonomy, and frozen manifest

The 12 source folders are mapped to three macro classes: healthy baseline, continuous friction/wear, and impulsive shocks/structural anomalies. This cell validates paths, split counts, class coverage, and overlap before any analysis or training.

In [ ]:
import pandas as pd
from src.split_manifest import load_split_manifest, rows_for_split

manifest = load_split_manifest("data/splits.csv", project_root=PROJECT_ROOT)
manifest_df = pd.DataFrame(manifest)
print(f"Total recordings: {len(manifest_df)}")
display(manifest_df.groupby(["split_multiclass_3class", "macro_class_index", "macro_class_name"]).size().rename("count").reset_index())
assert len(manifest_df) == 2148
assert manifest_df["split_multiclass_3class"].value_counts().to_dict() == {"train": 1504, "val": 322, "test": 322}
assert manifest_df["filepath"].is_unique
assert set(manifest_df["macro_class_index"]) == {0, 1, 2}
print("Manifest audit passed: 2,148 unique existing files; train/val/test = 1504/322/322; all classes present in every split.")

## 3. Run project unit tests

These tests cover the split-manifest contract, signal statistics, and zero-crossing-rate extractors.

In [ ]:
run(f"{sys.executable} -m unittest discover -s tests -v", cwd=PROJECT_ROOT)

## 4. Exploratory data analysis and preprocessing audit

EDA is intentionally performed before model fitting. It audits signal ranges, clipping/DC offset, statistical features, PCA/t-SNE structure, adjacent-file correlation, and representative waveform/spectrum/spectrogram plots.

In [ ]:
run(f"{sys.executable} eda_analysis.py", cwd=PROJECT_ROOT)
eda_images = sorted((PROJECT_ROOT / "results" / "visualizations").glob("eda_*.png"))
for image_path in eda_images:
    display(Markdown(f"### {image_path.name}"))
    display(Image(filename=str(image_path)))


## 5. Classical baselines

Before the deep model, the same frozen partitions are used for Logistic Regression, Random Forest, and XGBoost on handcrafted statistical features. `StandardScaler` is fit on training features only. The test results are reported for comparison, but never used to choose the neural architecture.

In [ ]:
run(f"{sys.executable} scripts/baseline_comparison_3class.py", cwd=PROJECT_ROOT)
for result_file in ["results/baseline_comparison_3class.csv", "results/baseline_comparison_3class.json"]:
    path = PROJECT_ROOT / result_file
    if path.exists():
        display(Markdown(f"### `{result_file}`"))
        if path.suffix == ".csv":
            display(pd.read_csv(path))
        else:
            display(Markdown("```json\n" + path.read_text() + "\n```"))


## 6. Train canonical TF-FaultNet

The training script loads the frozen manifest, preloads one-second waveforms, applies augmentation only to the training dataset, trains the in-model log-Mel + ResNet + squeeze-and-excitation network with cross-entropy label smoothing, checkpoints by lowest validation loss, and exports TorchScript/ONNX artifacts. The held-out test set is quarantined before training.

In [ ]:
run(f"{sys.executable} train_3class_pipeline.py --config configs/experiment_3class.json", cwd=PROJECT_ROOT)
history_path = PROJECT_ROOT / "results" / "training_convergence_history_3class.csv"
history = pd.read_csv(history_path)
display(history.tail())
best = history.loc[history["val_loss"].idxmin()]
print(f"Best validation checkpoint: epoch {int(best['epoch'])}, loss={best['val_loss']:.4f}, accuracy={best['val_acc']:.3%}, macro F1={best['val_f1']:.3%}")
for image_name in ["train_val_loss_convergence_3class.png", "validation_loss_curve_focused.png"]:
    image_path = PROJECT_ROOT / "results" / "visualizations" / image_name
    if image_path.exists():
        display(Image(filename=str(image_path)))


## 7. Locked held-out test evaluation

This is the first cell that evaluates the final checkpoint on the quarantined test partition. It reports waveform inference, spectrogram-backbone consistency, per-class precision/recall/F1/ROC-AUC/PR-AUC, confusion matrix, and ROC/PR curves.

In [ ]:
run(f"{sys.executable} evaluate_3class_test_set.py --config configs/experiment_3class.json", cwd=PROJECT_ROOT)
report_path = PROJECT_ROOT / "results" / "heldout_test_evaluation_report_3class.json"
report = json.loads(report_path.read_text())
display(pd.DataFrame(report["per_class"]))
display(pd.DataFrame([report["metrics"]]).T.rename(columns={0: "value"}))
for image_name in ["test_set_confusion_matrix_3class.png", "test_set_roc_pr_curves_3class.png", "test_sample_spectrograms_3class.png"]:
    image_path = PROJECT_ROOT / "results" / "visualizations" / image_name
    if image_path.exists():
        display(Image(filename=str(image_path)))


## 8. Optional architectural ablations

Ablations remove one component at a time (SE attention, residual connections, dual pooling, or instance normalization) using train/validation data only. Because this stage retrains multiple networks, it is disabled by default. Set `RUN_ABLATIONS = True` in the controls cell and rerun this cell when needed.

In [ ]:
if RUN_ABLATIONS:
    run(f"{sys.executable} ablation_study.py", cwd=PROJECT_ROOT)
    ablation_files = sorted((PROJECT_ROOT / "results" / "ablations").glob("*.csv"))
    for path in ablation_files:
        print(path)
        display(pd.read_csv(path))
else:
    print("Ablations skipped. Set RUN_ABLATIONS = True to execute them.")


## 9. Final artifact and metric summary

The following cell provides a compact reproducibility record and verifies that the expected checkpoint, report, and exported deployment models exist.

In [ ]:
expected = [
    "results/heldout_test_set_3class.npz",
    "results/checkpoints/best_tf_faultnet_3class.pt",
    "results/checkpoints/best_tf_faultnet_3class_jit.pt",
    "results/checkpoints/best_tf_faultnet_3class_backbone.onnx",
    "results/heldout_test_evaluation_report_3class.json",
    "results/test_roc_auc_metrics_3class.csv",
]
artifact_status = pd.DataFrame({"artifact": expected, "exists": [(PROJECT_ROOT / p).exists() for p in expected]})
display(artifact_status)
assert artifact_status["exists"].all(), "One or more required pipeline artifacts are missing."
print("Pipeline completed successfully.")


## 10. Download results from Colab

This creates a ZIP containing reports, plots, checkpoints, and exported models. For large files, prefer Google Drive mounting in the controls cell.

In [ ]:
bundle_path = WORKSPACE / "tf_faultnet_results.zip"
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in (PROJECT_ROOT / "results").rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(PROJECT_ROOT))
print(f"Created {bundle_path} ({bundle_path.stat().st_size / 1024 / 1024:.1f} MB)")
try:
    from google.colab import files
    files.download(str(bundle_path))
except ImportError:
    print("Not running in Colab; the bundle is available at", bundle_path)
